In [ ]:
from google.colab import files
uploaded = files.upload()

Saving markets_raw_unscored (1).csv to markets_raw_unscored (1).csv


In [ ]:
import pandas as pd
df = pd.read_csv('markets_raw_unscored (1).csv')
df.shape          # how many rows & columns
df.dtypes         # are dates strings or datetime?



,0
trade_id,object
client_id,object
trade_open_time,object
trade_close_time,object
client_country,object
ip_country,object
kyc_status,object
platform,object
leverage,int64
instrument,object


In [ ]:
df['trade_open_time']  = pd.to_datetime(df['trade_open_time'])
df['trade_close_time'] = pd.to_datetime(df['trade_close_time'])

In [ ]:
df[['trade_open_time','trade_close_time']].dtypes

,0
trade_open_time,datetime64[ns]
trade_close_time,datetime64[ns]


In [ ]:
df.head(1)


,trade_id,client_id,trade_open_time,trade_close_time,client_country,ip_country,kyc_status,platform,leverage,instrument,...,execution_latency_ms,slippage_pips,gross_pnl_usd,commission_usd,net_pnl_usd,account_balance_usd,margin_used_usd,equity_usd,margin_level_pct,max_drawdown_pct
0,TRD000001,CLT00414,2023-01-01 09:57:32,2023-01-01 11:03:32,Switzerland,Switzerland,Verified,MT4,100,GBPUSD,...,132.1,0.4,8506.74,33.86,8472.88,171082.86,23614.61,181549.73,768.8,56.95


In [ ]:
df['lot_size'].mean()

np.float64(10.68554)

In [ ]:
df['execution_latency_ms'].mean()

np.float64(83.86664999999999)

In [ ]:
df['slippage_pips'].mean()

np.float64(0.52157)

In [ ]:
# Flag suspicious trades:

import pandas as pd

# Defining the columns we want to monitor
metrics = ['lot_size', 'execution_latency_ms', 'slippage_pips', 'max_drawdown_pct']

# Calculating dynamic thresholds: Mean + (2 * Standard Deviation)
# Anything significantly above average becomes "suspicious"
thresholds = df[metrics].mean() + (2 * df[metrics].std())

# Creating the binary flag
# .any(axis=1) returns True if any single column exceeds its threshold
df['is_suspicious'] = (df[metrics] > thresholds).any(axis=1).astype(int)

print(df['is_suspicious'] )

0       0
1       1
2       1
3       0
4       0
       ..
1995    0
1996    0
1997    0
1998    0
1999    0
Name: is_suspicious, Length: 2000, dtype: int64


In [ ]:
df.shape

(2000, 26)

In [ ]:
# Detectinf IP mismatch. Will compare client_country vs ip_country. If they don't match, that's a fraud signal.
df['is_ip_mismatch'] = (
    df['client_country'].str.strip().str.upper() !=
    df['ip_country'].str.strip().str.upper()
).astype(int)

df['is_ip_mismatch']

,is_ip_mismatch
0,0
1,0
2,0
3,0
4,0
...,...
1995,0
1996,0
1997,0
1998,0


In [ ]:
#will see mismatch ips

mismatches = df[df['is_ip_mismatch'] == 1]
print(mismatches)

       trade_id client_id     trade_open_time    trade_close_time  \
12    TRD000013  CLT00092 2023-01-07 03:36:43 2023-01-07 04:04:43   
19    TRD000020  CLT00218 2023-01-08 03:03:32 2023-01-08 04:43:32   
20    TRD000021  CLT00156 2023-01-08 10:42:57 2023-01-08 10:56:57   
38    TRD000039  CLT00548 2023-01-16 08:55:46 2023-01-16 09:06:46   
56    TRD000057  CLT00176 2023-01-24 17:17:12 2023-01-24 17:51:12   
...         ...       ...                 ...                 ...   
1914  TRD001915  CLT00337 2024-11-23 08:06:14 2024-11-23 08:28:14   
1941  TRD001942  CLT00530 2024-12-06 07:26:09 2024-12-06 09:35:09   
1944  TRD001945  CLT00013 2024-12-07 14:29:19 2024-12-07 16:51:19   
1974  TRD001975  CLT00272 2024-12-22 01:36:27 2024-12-22 02:58:27   
1978  TRD001979  CLT00191 2024-12-22 09:53:22 2024-12-22 11:59:22   

      client_country      ip_country kyc_status   platform  leverage  \
12     United States          France   Verified        MT4       100   
19           Nigeria       

In [ ]:
df['is_unverified_kyc'] = (df['kyc_status']!= 'Verified').astype(int)
df['is_unverified_kyc']

,is_unverified_kyc
0,0
1,0
2,0
3,0
4,1
...,...
1995,1
1996,0
1997,1
1998,0


In [ ]:
df.shape

(2000, 28)

In [36]:
# Anomaly Score

import pandas as pd

# 1. Calculating Dynamic Risk Thresholds (Mean + 1 Std Dev)
# This defines what "100% risk" (1.0) looks like for  specific data
MAX_LOT = df['lot_size'].mean() + 2*df['lot_size'].std()
MAX_LATENCY = df['execution_latency_ms'].mean() + 2*df['execution_latency_ms'].std()
MAX_DRAWDOWN = df['max_drawdown_pct'].mean() + 2*df['max_drawdown_pct'].std()

# 2. Scaling and Clipping (0 to 1 range)
# Logic: If a value hits the threshold, it becomes 1. If it's double the threshold, it's still 1.
df['norm_lot'] = (df['lot_size'] / MAX_LOT).clip(0, 1)
df['norm_latency'] = (df['execution_latency_ms'] / MAX_LATENCY).clip(0, 1)
df['norm_drawdown'] = (df['max_drawdown_pct'] / MAX_DRAWDOWN).clip(0, 1)

# 3. Defining Weights (sum to 1.0)
weights = {
    'lot': 0.30,
    'lat': 0.10,
    'dd': 0.20,
    'ip': 0.25,
    'kyc': 0.15
}

# 4. Final Anomaly Score (0-100)
# 'is_ip_mismatch' and 'is_unverified_kyc' are already 0 or 1
df['anomaly_score'] = (
    (df['norm_lot'] * weights['lot']) +
    (df['norm_latency'] * weights['lat']) +
    (df['norm_drawdown'] * weights['dd']) +
    (df['is_ip_mismatch'] * weights['ip']) +
    (df['is_unverified_kyc'] * weights['kyc'])
).clip(0, 1) * 100

In [ ]:
df['anomaly_score']

,anomaly_score
0,31.823481
1,17.819931
2,25.773170
3,17.278359
4,29.423503
...,...
1995,35.340090
1996,25.335343
1997,49.387715
1998,8.292182


In [ ]:

 # Defining risk levels
 # 1. Defining custom thresholds (bins) and labels
# Bins: 0-30 (Low), 30-60 (Medium), 60-85 (High), 85-100 (Critical)
bins = [0, 30, 60, 85, 100]
labels = ['Low', 'Medium', 'High', 'Critical']

# 2. Create the risk_level column
df['risk_level'] = pd.cut(df['anomaly_score'], bins=bins, labels=labels, include_lowest=True)

# 3. Quick check of your distribution
print(df['risk_level'].value_counts())

risk_level
Low         1206
Medium       750
High          44
Critical       0
Name: count, dtype: int64


In [ ]:
df.shape


(2000, 33)

In [32]:
print(df['risk_level'].value_counts())
print(df['anomaly_score'].describe())

risk_level
Low         1206
Medium       750
High          44
Critical       0
Name: count, dtype: int64
count    2000.000000
mean       28.327430
std        13.555744
min         2.153070
25%        18.380768
50%        26.477120
75%        36.713506
max        82.867119
Name: anomaly_score, dtype: float64


In [41]:
import numpy as np
import pandas as pd

# 1. Deriving the trade duration in minutes
df['trade_duration_min'] = (df['trade_close_time'] - df['trade_open_time']).dt.total_seconds() / 60

# 2. Defining list of conditions (ordered by priority)
conditions = [
    (df['margin_level_pct'] < 100),                          # 1. Margin Call (Critical)
    (df['kyc_status'].isin(['Rejected', 'Expired'])),        # 2. KYC Breach
    (df['trade_duration_min'] < 1),                          # 3. Rapid Fire Trading
    (df['max_drawdown_pct'] > MAX_DRAWDOWN),                 # 4. Large Drawdown
    (df['lot_size'] > MAX_LOT),                             # 5. Unusual Volume
    (df['is_ip_mismatch'] == 1)                              # 6. IP Mismatch
]

# 3. Defining the corresponding labels
choices = [
    'Margin Call',
    'KYC Breach',
    'Rapid Fire Trading',
    'Large Drawdown',
    'Unusual Volume',
    'IP Mismatch'
]

# 4. Applying logic using np.select
# 'None' is the default if no conditions above are met
df['alert_type'] = np.select(conditions, choices, default='None')

# Check results
print(df[['alert_type', 'lot_size', 'margin_level_pct']].head(2000))


       alert_type  lot_size  margin_level_pct
0            None      4.89            768.80
1            None      8.16           1057.29
2            None      4.53           3992.66
3            None      2.37          25015.70
4            None      8.01           2154.02
...           ...       ...               ...
1995   KYC Breach      5.54           3817.83
1996         None      9.89           3090.87
1997   KYC Breach      7.76            139.29
1998  Margin Call      5.15            -61.05
1999   KYC Breach      0.28          25580.87

[2000 rows x 3 columns]
